# ChEMBL 기반 LINCS 약물-타깃 유전자 전처리

이 노트북은 LINCS metadata의 perturbation 이름(`cmap_name`)을 ChEMBL molecule과 매칭한 뒤, ChEMBL의 약물-타깃 정보를 이용해 각 sample의 target gene set과 `U′` binary mask를 생성한다.

## 최종 산출물

| 파일 | 설명 |
|---|---|
| `./chembl/metadata_X_with_chembl_targets.csv` | LINCS metadata에 ChEMBL ID, target gene, LINCS gene 위치 정보를 추가한 테이블 |
| `./chembl/U_prime_chembl_lincs_positions.npz` | sample별 target gene binary mask 행렬과 관련 metadata |

## 전체 처리 흐름

1. LINCS metadata와 ChEMBL SQLite DB 경로를 설정한다.
2. ChEMBL의 molecule preferred name 및 synonym을 수집한다.
3. 이름 정규화를 통해 `cmap_name`과 ChEMBL molecule을 매칭한다.
4. ChEMBL `drug_mechanism` 기반 human single-protein target gene을 추출한다.
5. target gene을 LINCS gene index로 변환한다.
6. 각 sample에 대해 `U′` binary mask를 생성하고 저장한다.


## 1. 라이브러리 로드

파일 경로 처리, 정규표현식 기반 이름 정규화, 행렬 생성, 테이블 처리, SQLite 연결에 필요한 라이브러리를 불러온다.


In [ ]:
import os
import re
import time
import numpy as np
import pandas as pd
import sqlite3

## 2. 경로 설정

입력 데이터와 출력 디렉터리를 정의한다.

- `LINCS_DIR`: LINCS metadata 파일 위치
- `OUTPUT_DIR`: ChEMBL 전처리 결과 저장 위치
- `DATA`: ChEMBL DB와 gene info 파일 위치


In [ ]:
LINCS_DIR = "./lincs/"
OUTPUT_DIR = "./chembl/"
DATA = "./data/"

METADATA = os.path.join(LINCS_DIR, "metadata_X.csv")
CHEMBL_TARGETS = os.path.join(OUTPUT_DIR, 'targets')
GENE_INFO = os.path.join(DATA,"geneinfo_beta.txt")
CHEMBL_DB = os.path.join(DATA, "chembl_36", "chembl_36_sqlite", "chembl_36.db")

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. LINCS metadata 로드

`metadata_X.csv`를 읽고, 기존 index column으로 저장된 `Unnamed: 0`을 `sample_id`로 변경한다. 이후 매칭에 필요한 주요 컬럼을 확인한다.


In [ ]:
metadata_X = pd.read_csv(METADATA, low_memory = False)
metadata_X = metadata_X.rename(columns={"Unnamed: 0": "sample_id"})
metadata_X[["sample_id", "pert_id", "cell_iname", "cmap_name"]].head()

## 4. ChEMBL SQLite DB 연결

ChEMBL SQLite database에 연결한다.


In [ ]:
conn = sqlite3.connect(CHEMBL_DB)

## 5. ChEMBL 약물 이름 및 synonym 수집

ChEMBL의 `molecule_dictionary`에서 preferred name을, `molecule_synonyms`에서 synonym을 가져와 molecule별 가능한 이름 후보를 만든다.


In [ ]:
drug_name_sql = """
SELECT
    md.chembl_id AS molecule_chembl_id,
    md.pref_name AS drug_name
FROM molecule_dictionary md
WHERE md.pref_name IS NOT NULL

UNION

SELECT
    md.chembl_id AS molecule_chembl_id,
    ms.synonyms AS drug_name
FROM molecule_dictionary md
JOIN molecule_synonyms ms
    ON md.molregno = ms.molregno
WHERE ms.synonyms IS NOT NULL
"""

chembl_drug_names = pd.read_sql_query(drug_name_sql, conn)
chembl_drug_names.head()

## 6. 약물 이름 정규화 함수 정의

대소문자, 공백, 특수문자 차이 때문에 생기는 매칭 실패를 줄이기 위해 이름을 소문자 알파뉴메릭 문자열로 정규화한다.


In [ ]:
def normalize_name(x):
    if pd.isna(x):
        return ""
    x = str(x).lower().strip()
    x = re.sub(r"[^a-z0-9]+", "", x)
    return x

## 7. ChEMBL 약물 이름 정규화

ChEMBL molecule name과 synonym에 정규화 컬럼을 추가하고, 빈 문자열 및 중복 행을 제거한다.


In [ ]:
chembl_drug_names["drug_name_norm"] = chembl_drug_names["drug_name"].apply(normalize_name)

chembl_drug_names = chembl_drug_names[
    chembl_drug_names["drug_name_norm"] != ""
].drop_duplicates()

chembl_drug_names.shape

## 8. LINCS `cmap_name`과 ChEMBL molecule 매칭

LINCS metadata의 `cmap_name`을 같은 방식으로 정규화한 뒤, ChEMBL name/synonym과 left join한다.


In [ ]:
metadata_X["cmap_name_norm"] = metadata_X["cmap_name"].apply(normalize_name)

metadata_with_chembl = metadata_X.merge(
    chembl_drug_names[["molecule_chembl_id", "drug_name", "drug_name_norm"]],
    left_on="cmap_name_norm",
    right_on="drug_name_norm",
    how="left",
)

## 9. sample 단위 중복 매칭 정리

하나의 sample에 여러 ChEMBL molecule이 매칭될 수 있으므로, `sample_id` 기준으로 정렬 후 첫 번째 매칭만 유지한다.


In [ ]:
metadata_with_chembl = (
    metadata_with_chembl
    .sort_values(["sample_id", "molecule_chembl_id"])
    .drop_duplicates(subset=["sample_id"], keep="first")
    .reset_index(drop=True)
)

## 10. ChEMBL 매칭률 확인

전체 sample 중 ChEMBL molecule ID가 매칭된 sample 수와 비율을 출력한다.


In [ ]:
print("Total samples:", len(metadata_with_chembl))
print("Mapped samples:", metadata_with_chembl["molecule_chembl_id"].notna().sum())
print("Mapping rate:", metadata_with_chembl["molecule_chembl_id"].notna().mean())

## 11. 미매칭 perturbation 확인

ChEMBL molecule로 매칭되지 않은 `pert_id`, `cmap_name` 조합을 확인한다.


In [ ]:
metadata_with_chembl[
    metadata_with_chembl["molecule_chembl_id"].isna()
][["pert_id", "cmap_name"]].drop_duplicates().head(30)

## 12. ChEMBL mechanism 기반 target gene 추출

ChEMBL `drug_mechanism` 테이블에서 human single-protein target에 해당하는 gene symbol을 추출한다.  
여기서는 mechanism 정보가 명확한 target만 사용한다.


In [ ]:
mechanism_sql = """
SELECT DISTINCT
    md.chembl_id AS molecule_chembl_id,
    td.chembl_id AS target_chembl_id,
    cs.component_synonym AS gene_symbol,
    dm.mechanism_of_action,
    dm.action_type
FROM drug_mechanism dm
JOIN molecule_dictionary md
    ON dm.molregno = md.molregno
JOIN target_dictionary td
    ON dm.tid = td.tid
JOIN target_components tc
    ON td.tid = tc.tid
JOIN component_synonyms cs
    ON tc.component_id = cs.component_id
WHERE td.organism = 'Homo sapiens'
  AND td.target_type = 'SINGLE PROTEIN'
  AND cs.syn_type = 'GENE_SYMBOL'
"""

mechanism_targets = pd.read_sql_query(mechanism_sql, conn)
mechanism_targets.head()

## 13. 선택적 activity fallback 코드

아래 코드는 `activities` 기반 target을 추가로 사용하기 위한 fallback 예시다. 현재 workflow에서는 mechanism 기반 target만 사용하므로 주석 처리되어 있다.


In [ ]:
'''activity_sql = """
SELECT DISTINCT
    md.chembl_id AS molecule_chembl_id,
    td.chembl_id AS target_chembl_id,
    cs.component_synonym AS gene_symbol
FROM activities act
JOIN assays ass
    ON act.assay_id = ass.assay_id
JOIN molecule_dictionary md
    ON act.molregno = md.molregno
JOIN target_dictionary td
    ON ass.tid = td.tid
JOIN target_components tc
    ON td.tid = tc.tid
JOIN component_synonyms cs
    ON tc.component_id = cs.component_id
WHERE td.organism = 'Homo sapiens'
  AND td.target_type = 'SINGLE PROTEIN'
  AND cs.syn_type = 'GENE_SYMBOL'
  AND act.pchembl_value >= 6
"""

activity_targets = pd.read_sql_query(activity_sql, conn)
activity_targets.head()'''

'''mech_mols = set(mechanism_targets["molecule_chembl_id"])

activity_fallback = activity_targets[
    ~activity_targets["molecule_chembl_id"].isin(mech_mols)
].copy()

chembl_targets = pd.concat(
    [
        mechanism_targets[["molecule_chembl_id", "gene_symbol"]],
        activity_fallback[["molecule_chembl_id", "gene_symbol"]],
    ],
    ignore_index=True,
).drop_duplicates()'''

## 14. molecule별 target gene list 생성

mechanism target에서 molecule-gene 중복을 제거한 뒤, molecule별 target gene list를 만든다.


In [ ]:
chembl_targets = mechanism_targets[["molecule_chembl_id", "gene_symbol"]].drop_duplicates()

chembl_target_lists = (
    chembl_targets
    .groupby("molecule_chembl_id")["gene_symbol"]
    .apply(lambda x: sorted(set(x)))
    .reset_index()
    .rename(columns={"gene_symbol": "chembl_target_genes"})
)

chembl_target_lists.head()

## 15. LINCS metadata에 target gene list 결합

ChEMBL molecule ID를 기준으로 target gene list를 metadata에 결합하고, target 개수를 계산한다.


In [ ]:
metadata_targets = metadata_with_chembl.merge(
    chembl_target_lists,
    on="molecule_chembl_id",
    how="left",
)

metadata_targets["chembl_target_genes"] = metadata_targets["chembl_target_genes"].apply(
    lambda x: x if isinstance(x, list) else []
)

metadata_targets["n_chembl_targets"] = metadata_targets["chembl_target_genes"].apply(len)

## 16. target 결합 결과 확인

sample별 ChEMBL molecule ID와 target gene list를 확인한다.


In [ ]:
metadata_targets[
    ["sample_id", "cell_iname", "cmap_name", "molecule_chembl_id", "chembl_target_genes", "n_chembl_targets"]
].head()

## 17. mechanism target mapping rate 확인

전체 sample 중 ChEMBL mechanism target을 하나 이상 가진 sample 수와 비율을 출력한다.


In [ ]:
print("Total samples:", len(metadata_targets))
print("Samples with mechanism targets:", (metadata_targets["n_chembl_targets"] > 0).sum())
print("Mechanism target mapping rate:", (metadata_targets["n_chembl_targets"] > 0).mean())

## 18. LINCS gene 정보 로드 및 gene symbol-position map 생성

`geneinfo_beta.txt`에서 LINCS gene ID와 gene symbol을 읽고, gene symbol을 `U′` vector 내 위치 index로 변환하기 위한 dictionary를 만든다.


In [ ]:
gene_info = pd.read_csv(GENE_INFO, sep="\t", low_memory=False)

gene_info_use = gene_info[["gene_id", "gene_symbol"]].dropna().copy()
gene_info_use["gene_id"] = gene_info_use["gene_id"].astype(int)
gene_info_use["gene_symbol"] = gene_info_use["gene_symbol"].astype(str)
gene_info_use = gene_info_use.reset_index(drop=True)

lincs_gene_ids = gene_info_use["gene_id"].to_numpy()
lincs_gene_symbols = gene_info_use["gene_symbol"].to_numpy()

symbol_to_pos = {
    symbol: idx
    for idx, symbol in enumerate(lincs_gene_symbols)
}

n_genes = len(lincs_gene_symbols)

## 19. target gene을 LINCS gene position으로 매핑하는 함수

각 ChEMBL target gene이 LINCS gene list에 존재하는 경우에만 gene symbol과 position을 유지한다.


In [ ]:
def map_targets_to_lincs_positions(target_genes):
    mapped_genes = []
    positions = []
    
    for gene in target_genes:
        if gene in symbol_to_pos:
            mapped_genes.append(gene)
            positions.append(symbol_to_pos[gene])
    
    return mapped_genes, positions

## 20. sample별 LINCS target position 계산

각 sample의 ChEMBL target gene list를 LINCS gene position list로 변환하고, 하나 이상의 LINCS target을 가진 sample만 남긴다.


In [ ]:
mapped = metadata_targets["chembl_target_genes"].apply(map_targets_to_lincs_positions)

metadata_targets["target_genes_lincs"] = mapped.apply(lambda x: x[0])
metadata_targets["target_positions_lincs"] = mapped.apply(lambda x: x[1])
metadata_targets["n_lincs_targets"] = metadata_targets["target_positions_lincs"].apply(len)

metadata_mapped = metadata_targets[
    metadata_targets["n_lincs_targets"] > 0
].copy()

metadata_mapped.shape

## 21. `U′` binary mask 생성 함수

`U′`는 길이가 LINCS gene 수인 binary vector다.  
target gene 위치는 1, 나머지 위치는 0으로 표시한다.


In [ ]:
def make_u_prime_mask(target_positions, n_genes):
    mask = np.zeros(n_genes, dtype=np.int8)
    mask[target_positions] = 1
    return mask

## 22. sample별 `U′` 행렬 생성

각 sample의 target position list를 binary mask로 변환하고, 모든 sample의 mask를 하나의 행렬로 쌓는다.


In [ ]:
metadata_mapped["U_prime"] = metadata_mapped["target_positions_lincs"].apply(
    lambda pos: make_u_prime_mask(pos, n_genes)
)

U_prime_matrix = np.vstack(metadata_mapped["U_prime"].values)

U_prime_matrix.shape

## 23. target metadata CSV 저장

list 형태 컬럼을 쉼표로 연결한 문자열로 변환한 뒤, 분석에 사용할 metadata CSV를 저장한다.


In [ ]:
metadata_save = metadata_mapped.drop(columns=["U_prime"]).copy()

for col in ["chembl_target_genes", "target_genes_lincs", "target_positions_lincs"]:
    metadata_save[col] = metadata_save[col].apply(
        lambda x: ",".join(map(str, x)) if isinstance(x, list) else ""
    )

metadata_save.to_csv(
    os.path.join(OUTPUT_DIR, "metadata_X_with_chembl_targets.csv"),
    index=False,
)

## 24. `U′` matrix와 관련 metadata 저장

`np.savez_compressed`를 사용해 `U_prime` 행렬과 sample/gene metadata를 압축 저장한다.


In [ ]:
np.savez_compressed(
    os.path.join(OUTPUT_DIR, "U_prime_chembl_lincs_positions.npz"),
    U_prime=U_prime_matrix,
    sample_ids=metadata_mapped["sample_id"].astype(str).to_numpy(),
    pert_ids=metadata_mapped["pert_id"].astype(str).to_numpy(),
    cmap_names=metadata_mapped["cmap_name"].astype(str).to_numpy(),
    cell_inames=metadata_mapped["cell_iname"].astype(str).to_numpy(),
    chembl_molecule_ids=metadata_mapped["molecule_chembl_id"].astype(str).to_numpy(),
    gene_ids=lincs_gene_ids,
    gene_symbols=lincs_gene_symbols,
)

## 25. 저장 결과 미리보기

저장된 metadata의 상위 3개 행을 확인한다.


In [ ]:
metadata_save.head(3)